# 用 PyGMT 畫自己的地圖
北市大課堂練習｜先用台灣，再延伸到世界。

**操作方式**：另存自己的副本，由上往下執行。每段先跑出圖，再做一個小修改。第 1–4 主題先不用 AI，第 5 主題再使用 Codex（前堂已安裝）或 Colab Gemini。

圖上的英文標籤可避免不同電腦中文字型問題；說明與練習使用中文。

## 先認識 GMT：用程式畫地圖

**GMT（Generic Mapping Tools）** 是一套地理資料處理與科學繪圖工具，可以畫海岸線、地形、地震分布、剖面與 3D 地形。

畫圖時，我們把幾個決定交給工具：**畫哪裡、用什麼資料、選什麼投影，以及如何用顏色與符號表達。** 保留程式後，就能換資料、改範圍、重新產生地圖。

今天的目標：從範例改出自己的地圖，知道各設定的意思，再用 AI 加快修改與延伸。

[GMT 官方網站](https://www.generic-mapping-tools.org/)

## 官方 Gallery：還可以畫什麼？

以下為 PyGMT 官方範例，點來源連結可查看圖片與完整程式。

### 3D 曲面：把網格數值立體呈現

這張使用數學函數產生的示範曲面，不是真實地形；同類繪圖方法也能呈現高程網格。

[來源：Plotting a surface](https://www.pygmt.org/v0.17.0/gallery/3d_plots/grdview_surface.html)

### 地震震源機制：用符號表達地震資訊

這些「沙灘球」符號用來表達震源機制，這堂課先欣賞，不需要學會解讀所有細節。

[來源：Focal mechanisms](https://www.pygmt.org/v0.17.0/gallery/seismology/meca.html)

### 分區設色：把統計資料放到地圖上

地圖也能呈現不同地區的統計數值，不只地形和地震。

[來源：Choropleth map](https://www.pygmt.org/v0.17.0/gallery/maps/choropleth_map.html)

圖像來源：PyGMT 官方 Gallery（Generic Mapping Tools 專案）；原始資料與製圖程式詳見各範例頁。

## Gallery：一張地圖可以變成什麼？

執行後面的繪圖程式，即可查看本教材的成果。

包含台灣海岸線、地震分布、彩色地形、3D 地形與全球地形。

同一份地形資料，連續改變觀看角度，就能製作旋轉動畫。圖中垂直尺度有誇大。

**想一想：你想畫哪個地方？希望讀者看到什麼？**

圖片由本教材 PyGMT 程式產生，動畫另用 Python 合成；後面會逐步拆解。也可探索 [PyGMT 官方 Gallery](https://www.pygmt.org/v0.17.0/gallery/index.html)。

## PyGMT：把 GMT 接進 Python 流程

PyGMT 讓我們用 **Python 呼叫 GMT**，將資料讀取、整理、分析和繪圖放在同一份 Notebook 裡，也方便使用變數與迴圈。

- **GMT**：提供地圖與科學繪圖能力。
- **PyGMT**：用 Python 操作 GMT，串接資料分析流程。
- **AI**：協助寫程式、修改與除錯；我們仍要檢查資料、範圍和圖例。

傳統 GMT 搭配 shell 也能寫變數與迴圈；PyGMT 的便利在於能接上 Python 生態系。

接下來先自己跑圖、改幾個參數，熟悉後再請 AI 延伸作品。

[PyGMT 官方文件](https://www.pygmt.org/v0.17.0/)

## 0. 準備環境
**Colab 請分開執行以下兩格。** 第一格安裝 Conda 後可能自動重啟；等重新連線，再執行第二格及後面的內容。勿在安裝期間重複按執行。

本機已裝好環境時會跳過安裝。此教材使用 PyGMT 0.17 / GMT 6.5；Colab 安裝流程仍需於課前用實際學生帳號確認。

In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.13"])
    import condacolab
    condacolab.install()
else:
    print("本機模式：使用目前 Python 環境。")

In [ ]:
import importlib.util
import subprocess
import sys

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if IN_COLAB:
    subprocess.check_call([
        "mamba", "install", "-y", "-c", "conda-forge",
        "pygmt=0.17", "gmt=6.5", "ghostscript=10.04", "pandas", "requests", "pillow"
    ])
else:
    print("跳過 Colab 安裝。")

In [ ]:
from pathlib import Path
from io import StringIO
import json
import math
import requests
import pandas as pd
import pygmt
from IPython.display import display, Image

OUT = Path("outputs")
DATA = Path("data")
OUT.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)
pygmt.config(FONT_TITLE="16p,Helvetica-Bold", FONT_LABEL="11p", FONT_ANNOT_PRIMARY="10p")
pygmt.show_versions()

def show_and_save(fig, filename):
    """存成 PNG，也在 Notebook 顯示同一張成果。"""
    path = OUT / filename
    fig.savefig(str(path), dpi=150, resize="+m0.3c")
    display(Image(filename=str(path)))

## 1. 台灣海岸線：第一張地圖
`region` 的順序是 **西、東、南、北**；`M15c` 表示麥卡托投影、圖寬 15 公分。每張圖都先建立新的 `Figure()`。

**試看看**：修改海洋顏色，或把東界從 123 改成 124，再重跑這一格。

In [ ]:
taiwan = [119, 123, 21, 26]
fig = pygmt.Figure()
fig.basemap(region=taiwan, projection="M15c", frame=["af", '+tTaiwan: coastlines'])
fig.coast(shorelines="0.6p,gray30", land="gray90", water="lightblue", resolution="h")
show_and_save(fig, "01_taiwan_coast.png")

## 地震資料從哪裡來？

PyGMT 負責繪圖，地震資料需要從資料服務取得。本次使用的是**地震目錄**：一列代表一筆事件，不是連續的地震波形。

| 資料來源 | 這堂課怎麼使用？ |
| --- | --- |
| **USGS（美國地質調查所）** | 本次主要地震資料來源；查詢全球事件並下載 CSV，方便從台灣延伸到其他地區。 |
| **台灣 GDMS（中央氣象署臺灣地震與地球物理資料管理系統）** | 補充介紹的台灣資料管道，包含地震目錄、波形與測站等資料；本次不操作下載。 |

先認識五個欄位：**經度、緯度、發生時間、規模、深度**。查詢時選定範圍、期間與最低規模，畫圖時再決定如何呈現。

不同目錄可能有不同的收錄條件與規模類型，筆數不一定相同；使用前要看欄位說明、單位與時間標準。

[USGS 查詢與 CSV 說明](https://earthquake.usgs.gov/fdsnws/event/1/) · [台灣 GDMS](https://gdms.cwa.gov.tw/) · [GDMS 使用說明](https://gdms.cwa.gov.tw/help.php)

## 2. 地震在哪裡？
使用 USGS 真實地震目錄，先下載 CSV，再用 PyGMT 畫點；不用 ObsPy。為方便比較，預設固定查詢 **2024-04-01 至 2024-05-01（UTC），規模 ≥ 4**，範圍與台灣地圖相同。

這是 USGS 目錄符合條件的事件，不代表所有台灣地震。原始 CSV 和查詢條件會存到 `data/`。若網路失敗，只在本機已有**相同查詢**備份時使用該備份，否則明確報錯。

In [ ]:
import hashlib

query = {
    "format": "csv", "starttime": "2024-04-01", "endtime": "2024-05-01",
    "minmagnitude": 4, "minlongitude": taiwan[0], "maxlongitude": taiwan[1],
    "minlatitude": taiwan[2], "maxlatitude": taiwan[3], "orderby": "time-asc"
}
endpoint = "https://earthquake.usgs.gov/fdsnws/event/1/query"
query_id = hashlib.sha256(json.dumps(query, sort_keys=True).encode()).hexdigest()[:12]
csv_path = DATA / f"usgs_{query_id}.csv"
try:
    response = requests.get(endpoint, params=query, timeout=90)
    response.raise_for_status()
    if not response.text.strip():
        raise ValueError("查詢無資料，請調整期間或範圍。")
    raw = pd.read_csv(StringIO(response.text))
    required = ["longitude", "latitude", "mag", "depth", "time"]
    if not set(required).issubset(raw.columns):
        raise ValueError("回傳資料缺少必要欄位。")
    csv_path.write_text(response.text, encoding="utf-8")
    print("已下載：", response.url)
except requests.RequestException:
    if not csv_path.exists():
        raise
    raw = pd.read_csv(csv_path)
    print("網路失敗，使用相同條件的備份：", csv_path)

(DATA / f"usgs_{query_id}_query.json").write_text(
    json.dumps({"endpoint": endpoint, "parameters": query}, indent=2), encoding="utf-8"
)
quakes = raw.dropna(subset=["longitude", "latitude", "mag", "depth"]).copy()
if quakes.empty:
    raise ValueError("沒有可繪製的事件。")
print(f"原始 {len(raw)} 筆；有效 {len(quakes)} 筆。深度單位：km；時間：UTC。")
display(quakes[["time", "longitude", "latitude", "mag", "depth"]].head())

### 先畫點，再讀大小與顏色
圓圈越大表示規模越大，顏色表示深度。圓圈直徑只是繪圖設定，**不是地震影響半徑**。

**試看看**：把前一格最低規模改成 5，重新下載並畫圖，比較筆數。

In [ ]:
def magnitude_size(magnitude):
    return 0.07 * (magnitude - 2)  # 圓圈直徑，cm；本練習使用 M >= 4

depth_max = max(20, math.ceil(quakes.depth.max() / 20) * 20)
fig = pygmt.Figure()
pygmt.makecpt(cmap="viridis", series=[0, depth_max, 1], continuous=True)
fig.basemap(region=taiwan, projection="M15c",
            frame=["af", "+tTaiwan earthquakes | USGS"])
fig.coast(land="gray95", water="aliceblue", shorelines="0.5p,gray40", resolution="h")
fig.plot(x=quakes.longitude, y=quakes.latitude,
         size=magnitude_size(quakes.mag), style="c",
         fill=quakes.depth, cmap=True, pen="0.25p,gray20")
for mag in [4, 5, 6, 7]:
    fig.plot(x=[taiwan[0]-10], y=[taiwan[2]-10],
             style=f"c{magnitude_size(mag)}c", fill="gray70",
             pen="0.25p,gray20", label=f"M {mag}")
fig.legend(position="JTL+jTL+o0.2c", box="+gwhite+p0.5p")
fig.colorbar(frame=["xaf", "y+lDepth (km)"])
print(f"UTC {query['starttime']} — {query['endtime']}, M >= {query['minmagnitude']}; N={len(quakes)}")
show_and_save(fig, "02_taiwan_earthquakes.png")

## 3. 彩色地形：高度變成顏色
`load_earth_relief()` 取得地形網格，`grdimage()` 把高度畫成顏色。`02m` 指 2 角分，**不是 2 公尺**。第一次執行需要下載 GMT 地形資料。

**試看看**：把 `shading=True` 改成 `False`，比較起伏的可讀性。深色也可能來自陰影，不能只用明暗判斷高低。

In [ ]:
grid = pygmt.datasets.load_earth_relief(
    resolution="02m", region=taiwan, registration="gridline"
)
print("地形網格：", grid.shape, "；高程單位：m")
fig = pygmt.Figure()
fig.grdimage(grid=grid, region=taiwan, projection="M15c",
             cmap="geo", shading=True, frame=["af", "+tTaiwan: land and seafloor"])
fig.coast(shorelines="0.5p,gray25", resolution="h")
fig.colorbar(frame=["xaf", "y+lElevation (m)"])
show_and_save(fig, "03_taiwan_relief.png")

## 4. 3D 地形：換個角度看台灣
用 `grdview()` 將同一份地形畫成斜視圖。`perspective=[方位角, 仰角]` 控制觀看方向，`zsize` 控制垂直尺寸。

圖中垂直方向為了辨認起伏而誇大，不能當成真實坡度。這是固定視角圖片，不是滑鼠可拖曳的模型。

**試看看**：把方位角 135 改為 225，仰角維持 35，觀察哪些山被遮住。

In [ ]:
azimuth = 135
elevation = 35
fig = pygmt.Figure()
fig.grdview(grid=grid, region=[*taiwan, -8000, 4000],
            projection="M15c", perspective=[azimuth, elevation],
            zsize="3c", surftype="s", cmap="geo",
            frame=["xaf", "yaf", "zaf+lElevation (m)", "+tTaiwan: 3D relief"])
show_and_save(fig, "04_taiwan_3d.png")

### 選做：旋轉動畫
這格會畫 12 個方向並合成 GIF，耗時比單張圖長。可先跳過，等核心練習完成再回來。圖面會隨視角裁切，合成時置中並統一畫布大小。

In [ ]:
RUN_ANIMATION = False  # 想試旋轉時改成 True

if RUN_ANIMATION:
    from PIL import Image as PILImage
    frames = []
    for angle in range(0, 360, 30):
        frame_fig = pygmt.Figure()
        frame_fig.grdview(grid=grid, region=[*taiwan, -8000, 4000],
                         projection="M12c", perspective=[angle, 35],
                         zsize="2.4c", surftype="s", cmap="geo",
                         frame=["xaf", "yaf", "zaf", "+tTaiwan relief"])
        path = OUT / f"rotation_{angle:03d}.png"
        frame_fig.savefig(str(path), dpi=100, resize="+m0.3c")
        with PILImage.open(path) as im:
            frames.append(im.convert("RGB"))
    width = max(im.width for im in frames)
    height = max(im.height for im in frames)
    canvases = []
    for im in frames:
        canvas = PILImage.new("RGB", (width, height), "white")
        canvas.paste(im, ((width-im.width)//2, (height-im.height)//2))
        canvases.append(canvas)
    gif_path = OUT / "taiwan_rotation.gif"
    canvases[0].save(gif_path, save_all=True, append_images=canvases[1:],
                     duration=250, loop=0)
    display(Image(filename=str(gif_path)))
else:
    print("動畫先跳過；把 RUN_ANIMATION 改成 True 即可執行。")

## 5. 從台灣到世界：開始用 AI
先執行以下全球地形範例，再用 Codex 或 Colab Gemini 協助修改。全球採 `01d`（1 度）資料，避免一開始下載太大的網格。

**練習**：選一個台灣以外的區域，請 AI 依原程式修改範圍、投影與解析度，保留資料來源和單位。

提示詞：
> 我在 Colab 使用 PyGMT 0.17，這段程式可以執行。請以它為基礎，改畫日本周邊的彩色地形，選合適的區域投影與資料解析度，加上標題與高程色階。不要加入 ObsPy。先解釋要改哪些參數，再給程式。

請核對：畫的是指定區域嗎？色階單位正確嗎？資料來源仍然存在嗎？能指出 AI 改了哪個參數嗎？

In [ ]:
world_grid = pygmt.datasets.load_earth_relief(resolution="01d", registration="gridline")
fig = pygmt.Figure()
fig.grdimage(grid=world_grid, region="g", projection="W15c",
             cmap="geo", shading=True, frame=["af", "+tWorld relief"])
fig.coast(shorelines="0.25p,gray30", resolution="c")
fig.colorbar(frame=["xaf", "y+lElevation (m)"])
show_and_save(fig, "05_world_relief.png")

## 6. 隔週繳交
選擇全球或自選區域，將 Notebook、成果圖與簡短說明放到 GitHub，交 repository 網址。說明繪圖區域、資料來源、主要觀察，以及 AI 協助了什麼。

- Notebook：儲存目前 `.ipynb`，保留有用的執行輸出。
- 成果圖：在左側檔案區找到 `outputs/` 下載。
- 原始地震 CSV 與條件：在 `data/`。
- GitHub：建立 repository，上傳 Notebook、成果與 README，確認教師可讀取連結。

下格打包圖片與資料，方便下載；Notebook 請另外由 Colab 的檔案選單下載。

In [ ]:
import shutil
bundle = Path("submission_assets")
bundle.mkdir(exist_ok=True)
for folder in [OUT, DATA]:
    shutil.copytree(folder, bundle / folder.name, dirs_exist_ok=True)
archive = shutil.make_archive("submission_assets", "zip", root_dir=bundle)
print("已建立：", archive)

## 資料來源與版本
- [原始課程參考 Notebook](https://github.com/oceanicdayi/plot_plate_boundary_pygmt/blob/main/pygmt_plot_plate_boundary.ipynb)
- [USGS 地震目錄 API](https://earthquake.usgs.gov/fdsnws/event/1/)：真實事件資料；查詢條件另存 JSON。
- [GMT 全球地形資料](https://docs.generic-mapping-tools.org/latest/datasets/remote-data.html)：PyGMT 載入，首次使用需連網。
- [PyGMT 0.17 安裝文件](https://www.pygmt.org/v0.17.0/install.html)

本機執行驗證與 Colab 雲端安裝驗證是兩件事；實際測試結果請參閱同資料夾 README。